In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [1]:
pass  # Colab only: from google.colab import drive
pass  # Colab only: drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import requests, zipfile, io, os
import pandas as pd
import numpy as np

BASE_DIR     = DATA_DIR + "/"
DATASETS_DIR = BASE_DIR + "datasets/"
NATIONAL_DIR = BASE_DIR + "national/"
MODELS_DIR   = BASE_DIR + "models/"

AADF_PATH = DATASETS_DIR + "aadf_major_roads.csv"

# Download
if not os.path.exists(AADF_PATH):
    os.makedirs(DATASETS_DIR, exist_ok=True)

    url = "https://storage.googleapis.com/dft-statistics/road-traffic/downloads/data-gov-uk/dft_traffic_counts_aadf.zip"
    print(f"Downloading AADF data...")
    r = requests.get(url, timeout=180)
    print(f"Status: {r.status_code}, Size: {len(r.content)/1e6:.1f}MB")

    z        = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = [f for f in z.namelist() if f.endswith('.csv') and not f.startswith('__')][0]
    print(f"Extracting: {csv_name}")

    with z.open(csv_name) as src, open(AADF_PATH, 'wb') as dst:
        dst.write(src.read())
    print(f" Saved as aadf_major_roads.csv")
else:
    print(f" AADF file already exists")

#  Inspect
aadf = pd.read_csv(AADF_PATH, low_memory=False)
print(f"\nTotal records: {len(aadf):,}")
print(f"Columns: {list(aadf.columns)}")
print(f"\nYear range: {aadf['year'].min()} → {aadf['year'].max()}")
print(f"\nRoad types: {aadf['road_type'].value_counts().to_dict()}")
print(f"\nSample rows:")
print(aadf.head(3).to_string())

Status: 200, Size: 18.9MB
Extracting: dft_traffic_counts_aadf.csv
 Saved as aadf_major_roads.csv

Total records: 578,217
Columns: ['count_point_id', 'year', 'region_id', 'region_name', 'region_ons_code', 'local_authority_id', 'local_authority_name', 'local_authority_code', 'road_name', 'road_category', 'road_type', 'start_junction_road_name', 'end_junction_road_name', 'easting', 'northing', 'latitude', 'longitude', 'link_length_km', 'link_length_miles', 'estimation_method', 'estimation_method_detailed', 'pedal_cycles', 'two_wheeled_motor_vehicles', 'cars_and_taxis', 'buses_and_coaches', 'LGVs', 'HGVs_2_rigid_axle', 'HGVs_3_rigid_axle', 'HGVs_4_or_more_rigid_axle', 'HGVs_3_or_4_articulated_axle', 'HGVs_5_articulated_axle', 'HGVs_6_articulated_axle', 'all_HGVs', 'all_motor_vehicles']

Year range: 2000 → 2024

Road types: {'Major': 447926, 'Minor': 130291}

Sample rows:
   count_point_id  year  region_id region_name region_ons_code  local_authority_id local_authority_name local_authority_

In [3]:
from sklearn.neighbors import BallTree
from sklearn.preprocessing import MinMaxScaler

In [4]:
#Filter to most recent year and major roads only
latest_year = aadf['year'].max()
aadf_latest = aadf[
    (aadf['year'] == latest_year) &
    (aadf['road_type'] == 'Major')
].copy()

print(f"Latest year: {latest_year}")
print(f"Major road count points: {len(aadf_latest):,}")


Latest year: 2024
Major road count points: 17,940


In [5]:
#Compute volume features
#Total daily vehicle flow
aadf_latest['aadf_total'] = aadf_latest['all_motor_vehicles']

# HGV percentage — strong severity predictor
aadf_latest['hgv_pct'] = (
    aadf_latest['all_HGVs'] /
    aadf_latest['all_motor_vehicles'].replace(0, np.nan)
).fillna(0).clip(0, 1)

# Log volume — reduces skew from motorways dominating
aadf_latest['log_aadf'] = np.log1p(aadf_latest['aadf_total'])

# Drop rows with missing coordinates
aadf_latest = aadf_latest.dropna(subset=['latitude','longitude'])
print(f"After dropping missing coords: {len(aadf_latest):,}")

print(f"\nVolume distribution:")
print(aadf_latest['aadf_total'].describe())
print(f"\nHGV % distribution:")
print(aadf_latest['hgv_pct'].describe())

After dropping missing coords: 17,940

Volume distribution:
count     17940.000000
mean      21356.997938
std       23443.112396
min           0.000000
25%        8396.000000
50%       14274.000000
75%       24039.500000
max      210436.000000
Name: aadf_total, dtype: float64

HGV % distribution:
count    17940.000000
mean         0.040850
std          0.037934
min          0.000000
25%          0.016365
50%          0.028650
75%          0.051881
max          0.485958
Name: hgv_pct, dtype: float64


In [6]:
# Load national grid
grid = pd.read_csv(NATIONAL_DIR + 'national_grid_history.csv')
print(f"\nNational grid cells: {len(grid):,}")

# Spatial join — snap each grid cell to nearest AADF count point
print("\nSpatially joining AADF to national grid...")

aadf_coords_rad  = np.radians(aadf_latest[['latitude','longitude']].values)
grid_coords_rad  = np.radians(grid[['avg_lat','avg_lon']].values)

tree = BallTree(aadf_coords_rad, metric='haversine')
distances, indices = tree.query(grid_coords_rad, k=1)
dist_km = distances[:, 0] * 6371.0

grid['aadf_total']    = aadf_latest.iloc[indices[:, 0]]['aadf_total'].values
grid['log_aadf']      = aadf_latest.iloc[indices[:, 0]]['log_aadf'].values
grid['hgv_pct']       = aadf_latest.iloc[indices[:, 0]]['hgv_pct'].values
grid['road_name']     = aadf_latest.iloc[indices[:, 0]]['road_name'].values
grid['aadf_dist_km']  = dist_km.round(3)

# Flag cells too far from any count point (>5km — likely minor roads)
# For these, using OSM road hierarchy proxy
grid['aadf_matched']  = (dist_km <= 5.0).astype(int)

# For unmatched cells (minor roads), assign a low default volume
median_minor = 2000
grid.loc[grid['aadf_matched'] == 0, 'aadf_total'] = median_minor
grid.loc[grid['aadf_matched'] == 0, 'log_aadf']   = np.log1p(median_minor)
grid.loc[grid['aadf_matched'] == 0, 'hgv_pct']    = 0.03

print(f"\nGrid cells matched to AADF (within 5km): {grid['aadf_matched'].sum():,} "
      f"({grid['aadf_matched'].mean()*100:.1f}%)")
print(f"Grid cells using minor road proxy:       {(grid['aadf_matched']==0).sum():,}")



National grid cells: 43,903

Spatially joining AADF to national grid...

Grid cells matched to AADF (within 5km): 41,429 (94.4%)
Grid cells using minor road proxy:       2,474


In [7]:
# Compute exposure-adjusted accident rate
# Accidents per million vehicle kilometres — the gold standard safety metric
# vehicle_km_per_year = aadf * 365 * avg segment length (approx 200m = 0.2km)
grid['vehicle_km_per_year'] = grid['aadf_total'] * 365 * 0.2
grid['acc_per_mvkm'] = (
    grid['accidents_per_year'] /
    (grid['vehicle_km_per_year'] / 1_000_000).replace(0, np.nan)
).fillna(0)

# Cap extreme outliers at 99th percentile
cap = grid['acc_per_mvkm'].quantile(0.99)
grid['acc_per_mvkm'] = grid['acc_per_mvkm'].clip(upper=cap)

print(f"\nAccidents per million vehicle-km distribution:")
print(grid['acc_per_mvkm'].describe())


Accidents per million vehicle-km distribution:
count    43903.000000
mean         1.599236
std          2.666589
min          0.000000
25%          0.275294
50%          0.677814
75%          1.621140
max         17.534655
Name: acc_per_mvkm, dtype: float64


In [8]:
#Normalise new features
scaler = MinMaxScaler()
grid['log_aadf_norm']    = scaler.fit_transform(grid[['log_aadf']])
grid['hgv_pct_norm']     = scaler.fit_transform(grid[['hgv_pct']])
grid['acc_per_mvkm_norm']= scaler.fit_transform(grid[['acc_per_mvkm']])

In [9]:
#Sample — top 10 highest exposure-adjusted risk cells
print(f"\nTop 10 highest acc_per_mvkm (true risk) cells:")
print(grid.nlargest(10, 'acc_per_mvkm')[[
    'grid_id','region_name','total_accidents','aadf_total',
    'acc_per_mvkm','hgv_pct','road_name'
]].to_string())


Top 10 highest acc_per_mvkm (true risk) cells:
       grid_id     region_name  total_accidents  aadf_total  acc_per_mvkm   hgv_pct road_name
0  51.51_-0.13  Greater_London              521        8002     17.534655  0.031492      A400
1  51.52_-0.16  Greater_London              405       13262     17.534655  0.022847     A4380
2  51.54_-0.14  Greater_London              374       21908     17.534655  0.020860      A503
3  51.52_-0.06  Greater_London              370       20381     17.534655  0.024533       A11
4  51.52_-0.08  Greater_London              354       14112     17.534655  0.060020       A10
5  51.51_-0.14  Greater_London              351       13999     17.534655  0.021644        A4
6  51.52_-0.17  Greater_London              348       26655     17.534655  0.018533        A5
7  51.52_-0.07  Greater_London              346       18699     17.534655  0.042088       A11
8    51.5_-0.1  Greater_London              338        9143     17.534655  0.068358     A3202
9  51.58_-0.

In [11]:
#Save
grid.to_csv(NATIONAL_DIR + 'national_grid_with_volume.csv', index=False)
print(f"\n Saved national_grid_with_volume.csv ({len(grid):,} grid cells)")


 Saved national_grid_with_volume.csv (43,903 grid cells)
